<a href="https://colab.research.google.com/github/sonu786786/Fundamentals-of-Artificial-Intelligence/blob/main/Lab_06/Jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# LAB ASSIGNMENT 6 — PART 1: Min-Max Algorithm

In [2]:
import math

# ── Board utilities ─────────────────────────────────────────
def print_board(board):
    for i in range(0, 9, 3):
        print(" | ".join(board[i:i+3]))
        if i < 6: print("---------")

def check_winner(board, player):
    wins = [(0,1,2),(3,4,5),(6,7,8),
            (0,3,6),(1,4,7),(2,5,8),
            (0,4,8),(2,4,6)]
    return any(board[a]==board[b]==board[c]==player for a,b,c in wins)

def is_draw(board):
    return "." not in board

def evaluate(board):
    """Terminal utility: +10 if X wins, -10 if O wins, 0 draw."""
    if check_winner(board, "X"): return  10
    if check_winner(board, "O"): return -10
    return 0

# ── Min-Max core ─────────────────────────────────────────────
nodes_evaluated = 0          # global counter

def minimax(board, depth, is_maximizing):
    global nodes_evaluated
    nodes_evaluated += 1

    score = evaluate(board)
    if score != 0:           # terminal: win/loss
        return score - depth if score > 0 else score + depth
    if is_draw(board): return 0

    if is_maximizing:            # X's turn → maximize
        best = -math.inf
        for i in range(9):
            if board[i] == ".":
                board[i] = "X"
                best = max(best, minimax(board, depth+1, False))
                board[i] = "."
        return best
    else:                        # O's turn → minimize
        best = math.inf
        for i in range(9):
            if board[i] == ".":
                board[i] = "O"
                best = min(best, minimax(board, depth+1, True))
                board[i] = "."
        return best

def best_move_minimax(board):
    global nodes_evaluated
    nodes_evaluated = 0
    best_val, move = -math.inf, -1
    for i in range(9):
        if board[i] == ".":
            board[i] = "X"
            val = minimax(board, 0, False)
            board[i] = "."
            if val > best_val:
                best_val, move = val, i
    return move, nodes_evaluated

# ── Driver ────────────────────────────────────────────────────
if __name__ == "__main__":
    board = list(".........")
    print("Initial board:"); print_board(board)
    move, nodes = best_move_minimax(board)
    print(f"\nBest move for X (Maximizer): position {move}")
    print(f"Nodes evaluated by Min-Max : {nodes}")

Initial board:
. | . | .
---------
. | . | .
---------
. | . | .

Best move for X (Maximizer): position 0
Nodes evaluated by Min-Max : 549945


In [3]:
# LAB ASSIGNMENT 6 — PART 2: Alpha-Beta Pruning

In [4]:
import math, time

# (board helpers same as Part 1 — reused here)
# ── Alpha-Beta core ──────────────────────────────────────────
ab_nodes = 0          # nodes evaluated WITH pruning
pruned_count = 0      # branches pruned

def alphabeta(board, depth, alpha, beta, is_maximizing):
    global ab_nodes, pruned_count
    ab_nodes += 1

    score = evaluate(board)
    if score != 0:
        return score - depth if score > 0 else score + depth
    if is_draw(board): return 0

    if is_maximizing:
        best = -math.inf
        for i in range(9):
            if board[i] == ".":
                board[i] = "X"
                val = alphabeta(board, depth+1, alpha, beta, False)
                board[i] = "."
                best = max(best, val)
                alpha = max(alpha, best)
                if beta <= alpha:        # β-cutoff: minimizer won't allow
                    pruned_count += 1
                    break
        return best
    else:
        best = math.inf
        for i in range(9):
            if board[i] == ".":
                board[i] = "O"
                val = alphabeta(board, depth+1, alpha, beta, True)
                board[i] = "."
                best = min(best, val)
                beta = min(beta, best)
                if beta <= alpha:        # α-cutoff: maximizer won't allow
                    pruned_count += 1
                    break
        return best

def best_move_alphabeta(board):
    global ab_nodes, pruned_count
    ab_nodes, pruned_count = 0, 0
    best_val, move = -math.inf, -1
    for i in range(9):
        if board[i] == ".":
            board[i] = "X"
            val = alphabeta(board, 0, -math.inf, math.inf, False)
            board[i] = "."
            if val > best_val:
                best_val, move = val, i
    return move, ab_nodes, pruned_count

# ── Comparison driver ─────────────────────────────────────────
if __name__ == "__main__":
    board = list(".........")   # empty board → hardest case

    t0 = time.perf_counter()
    mm_move, mm_nodes = best_move_minimax(board)
    mm_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    ab_move, ab_nodes_cnt, pruned = best_move_alphabeta(board)
    ab_time = time.perf_counter() - t0

    reduction = (1 - ab_nodes_cnt / mm_nodes) * 100
    print(f"{'Algorithm':20} {'Nodes':>8} {'Time(s)':>10} {'Move':>6}")
    print("-"*48)
    print(f"{'Min-Max':20} {mm_nodes:>8} {mm_time:>10.5f} {mm_move:>6}")
    print(f"{'Alpha-Beta':20} {ab_nodes_cnt:>8} {ab_time:>10.5f} {ab_move:>6}")
    print(f"\nNodes reduced by {reduction:.1f}%  |  {pruned} branches pruned")
    print(f"Same optimal move: {mm_move == ab_move}")


Algorithm               Nodes    Time(s)   Move
------------------------------------------------
Min-Max                549945    1.89440      0
Alpha-Beta              34202    0.10141      0

Nodes reduced by 93.8%  |  13124 branches pruned
Same optimal move: True
